# 01 Market Selector

Refactors the pilot notebook market retrieval, filtering, inclusion/exclusion, and relevant-market output logic into a file-based stage. The original pilot notebook is not modified.


## Setup
Load config, locate the Falnama project root, and create repository folders.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "falnama_config.yaml").exists() or (candidate / "polymarket_geopolitics_anomaly_detection_pilot.ipynb").exists():
            return candidate
    raise RuntimeError("Could not locate Falnama project root. Run from inside the Falnama folder.")

PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config" / "falnama_config.yaml"
RUN_TIME_UTC = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def load_config() -> dict[str, Any]:
    with CONFIG_PATH.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

CONFIG = load_config()
REPOSITORIES = CONFIG.get("repositories", {})
for repo_rel in REPOSITORIES.values():
    (PROJECT_ROOT / repo_rel).mkdir(parents=True, exist_ok=True)

RUN_LOG_DIR = PROJECT_ROOT / REPOSITORIES.get("run_logs", "repositories/run_logs")
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)


def repo_path(key: str, default: str) -> Path:
    path = PROJECT_ROOT / REPOSITORIES.get(key, default)
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_run_log(notebook_name: str, records: list[dict[str, Any]]) -> Path:
    path = RUN_LOG_DIR / f"{notebook_name}_{RUN_TIME_UTC.replace(':', '').replace('-', '')}.jsonl"
    with path.open("x", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps({"run_time_utc": RUN_TIME_UTC, **record}, default=str) + "\n")
    return path


def load_csv_nonempty(path: Path) -> pd.DataFrame | None:
    if not path.exists() or path.stat().st_size <= 1:
        return None
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return None
    return df if not df.empty else None


def normalize_timestamp(value: Any) -> pd.Timestamp:
    if value is None or value == "" or (isinstance(value, float) and np.isnan(value)):
        return pd.NaT
    if isinstance(value, pd.Timestamp):
        return value.tz_localize("UTC") if value.tzinfo is None else value.tz_convert("UTC")
    if isinstance(value, (int, float, np.integer, np.floating)):
        unit = "ms" if float(value) > 10_000_000_000 else "s"
        return pd.to_datetime(value, unit=unit, utc=True, errors="coerce")
    return pd.to_datetime(value, utc=True, errors="coerce")


def parse_jsonish(value: Any, default: Any = None) -> Any:
    if default is None:
        default = []
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return default
        try:
            return json.loads(text)
        except Exception:
            return value
    return value


def first_present(obj: dict[str, Any] | pd.Series, keys: list[str], default: Any = None) -> Any:
    for key in keys:
        if key in obj and obj[key] not in (None, "") and not (isinstance(obj[key], float) and np.isnan(obj[key])):
            return obj[key]
    return default


def safe_slug(value: Any, fallback: str = "unknown") -> str:
    text = str(value if value not in (None, "") else fallback).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text[:100] or fallback


## Selector Functions
Adapted from the pilot notebook: market text normalization, keyword inclusion, hard rejection, optional Gamma API fetch, legacy-output fallback, and mock fallback.


In [2]:
import requests

SELECTOR_CFG = CONFIG.get("selector", {})
LEGACY = CONFIG.get("legacy_inputs", {})
RELEVANT_DIR = repo_path("relevant_markets", "repositories/relevant_markets")
LOGS: list[dict[str, Any]] = []

GAMMA_BASE_URL = "https://gamma-api.polymarket.com"
USER_AGENT = "falnama-market-selector/0.2 read-only research notebook"

GEOPOLITICS_KEYWORDS = SELECTOR_CFG.get("geopolitics_keywords", [])
HARD_REJECT_KEYWORDS = SELECTOR_CFG.get("hard_reject_keywords", [])
HARD_REJECT_TAGS = SELECTOR_CFG.get("hard_reject_tags", [])


def text_blob_from_market(market: dict[str, Any]) -> str:
    pieces: list[str] = []
    for key in ["question", "title", "description", "category", "slug", "eventSlug", "event_slug", "market_name"]:
        value = market.get(key)
        if value:
            pieces.append(str(value))
    tags = parse_jsonish(market.get("tags"), default=[])
    if isinstance(tags, list):
        for tag in tags:
            if isinstance(tag, dict):
                pieces.extend(str(tag.get(k, "")) for k in ["label", "name", "slug"])
            else:
                pieces.append(str(tag))
    events = parse_jsonish(market.get("events"), default=[])
    if isinstance(events, list):
        for event in events:
            if isinstance(event, dict):
                pieces.extend(str(event.get(k, "")) for k in ["title", "slug", "description"])
    return " ".join(pieces).lower()


def keyword_match_market(market: dict[str, Any]) -> bool:
    blob = text_blob_from_market(market)
    return any(re.search(rf"\b{re.escape(keyword.lower())}\b", blob) for keyword in GEOPOLITICS_KEYWORDS)


def hard_reject_market(market: dict[str, Any]) -> bool:
    blob = text_blob_from_market(market)
    if any(re.search(rf"\b{re.escape(term.lower())}\b", blob) for term in HARD_REJECT_KEYWORDS):
        return True
    tag_blob = str(market.get("tags", "")).lower()
    return any(re.search(rf"\b{re.escape(tag.lower())}\b", tag_blob) for tag in HARD_REJECT_TAGS)


def normalize_market_record(market: dict[str, Any]) -> dict[str, Any]:
    market_id = first_present(market, ["market_id", "id", "marketId"])
    slug = first_present(market, ["market_slug", "slug"])
    event_slug = first_present(market, ["event_slug", "eventSlug"])
    question = first_present(market, ["market_name", "question", "title"], "Unknown market")
    url = first_present(market, ["market_url", "url"])
    if not url and slug:
        url = f"https://polymarket.com/event/{event_slug}/{slug}" if event_slug else f"https://polymarket.com/market/{slug}"
    return {
        "market_id": None if pd.isna(market_id) else str(market_id),
        "market_slug": None if pd.isna(slug) else str(slug),
        "market_name": str(question),
        "event_id": first_present(market, ["event_id", "eventId"]),
        "event_slug": event_slug,
        "condition_id": first_present(market, ["condition_id", "conditionId"]),
        "description": first_present(market, ["description"], None),
        "category": first_present(market, ["category"], None),
        "tags": first_present(market, ["tags"], None),
        "volume": first_present(market, ["volume"], None),
        "liquidity": first_present(market, ["liquidity"], None),
        "start_date": first_present(market, ["start_date", "startDate"], None),
        "close_time": first_present(market, ["close_time", "endDate", "end_date"], None),
        "market_url": url,
        "source": first_present(market, ["source"], "selector"),
    }


def fetch_gamma_markets() -> pd.DataFrame:
    if not SELECTOR_CFG.get("enable_network_fetch", False):
        LOGS.append({"event": "network_fetch_skipped", "reason": "selector.enable_network_fetch is false"})
        return pd.DataFrame()
    rows: list[dict[str, Any]] = []
    max_markets = int(SELECTOR_CFG.get("max_markets", 500))
    tag_ids = SELECTOR_CFG.get("geopolitics_tag_ids") or [None]
    for tag_id in tag_ids:
        offset = 0
        while len(rows) < max_markets:
            params = {"closed": str(SELECTOR_CFG.get("closed_only", True)).lower(), "limit": 100, "offset": offset, "include_tag": "true"}
            if tag_id:
                params["tag_id"] = tag_id
            response = requests.get(f"{GAMMA_BASE_URL}/markets", params=params, headers={"User-Agent": USER_AGENT}, timeout=30)
            response.raise_for_status()
            payload = response.json()
            records = payload if isinstance(payload, list) else payload.get("markets", payload.get("data", []))
            if not records:
                break
            rows.extend(records)
            offset += len(records)
            time.sleep(0.2)
            if len(records) < 100:
                break
    return pd.DataFrame([normalize_market_record(r) for r in rows[:max_markets]])


def markets_from_legacy_files() -> pd.DataFrame:
    candidates: list[pd.DataFrame] = []
    legacy_markets = load_csv_nonempty(PROJECT_ROOT / LEGACY.get("legacy_markets", ""))
    if legacy_markets is not None:
        legacy_markets = legacy_markets.rename(columns={"question": "market_name", "slug": "market_slug", "url": "market_url"})
        legacy_markets["source"] = "legacy_market_file"
        candidates.append(legacy_markets)
    for key in ["legacy_ranked_anomalies", "legacy_strong_anomalies"]:
        df = load_csv_nonempty(PROJECT_ROOT / LEGACY.get(key, ""))
        if df is not None:
            df = df.rename(columns={"question": "market_name", "market_url": "market_url"})
            if "market_slug" not in df.columns:
                df["market_slug"] = df.get("market_name", pd.Series(dtype=str)).map(lambda x: safe_slug(x, "market"))
            df["source"] = key
            candidates.append(df)
    if not candidates:
        return pd.DataFrame()
    raw = pd.concat(candidates, ignore_index=True, sort=False)
    records = [normalize_market_record(row.to_dict()) for _, row in raw.iterrows()]
    out = pd.DataFrame(records)
    return out.drop_duplicates(subset=["market_id", "market_slug", "market_name"], keep="first").reset_index(drop=True)


def mock_relevant_markets() -> pd.DataFrame:
    return pd.DataFrame([
        {
            "market_id": "mock-market-001",
            "market_slug": "mock-geopolitical-risk-market",
            "market_name": "Mock geopolitical escalation market for smoke testing",
            "event_id": "mock-event-001",
            "event_slug": "mock-event",
            "condition_id": "mock-condition-001",
            "description": "Clearly labeled mock market used only when no real selector inputs exist.",
            "category": "geopolitics",
            "tags": "geopolitics|mock",
            "volume": 100000,
            "liquidity": 10000,
            "start_date": None,
            "close_time": None,
            "market_url": None,
            "source": "mock_selector_input",
        }
    ])


def filter_relevant_markets(markets: pd.DataFrame) -> pd.DataFrame:
    if markets.empty:
        return markets
    retained = []
    rejected = []
    for _, row in markets.iterrows():
        record = row.to_dict()
        reason = None
        if hard_reject_market(record):
            reason = "hard reject keyword or tag"
        elif not keyword_match_market(record):
            reason = "no geopolitics keyword confirmation"
        if reason:
            rejected.append({"market_name": record.get("market_name"), "market_id": record.get("market_id"), "reason": reason})
        else:
            retained.append(record)
    LOGS.append({"event": "selector_filter", "input_rows": len(markets), "retained_rows": len(retained), "rejected_rows": len(rejected)})
    if rejected:
        rejected_path = RELEVANT_DIR / f"rejected_selector_markets_{RUN_TIME_UTC.replace(':', '').replace('-', '')}.csv"
        pd.DataFrame(rejected).to_csv(rejected_path, index=False)
    return pd.DataFrame(retained)


def write_relevant_markets(df: pd.DataFrame) -> tuple[Path, Path]:
    stamp = RUN_TIME_UTC.replace(":", "").replace("-", "")
    csv_path = RELEVANT_DIR / f"relevant_markets_{stamp}.csv"
    json_path = RELEVANT_DIR / f"relevant_markets_{stamp}.json"
    df.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(df.where(pd.notna(df), None).to_dict(orient="records"), indent=2, default=str), encoding="utf-8")
    return csv_path, json_path


## Run Selector
Writes timestamped CSV and JSON relevant-market files to `repositories/relevant_markets/`.


In [3]:
markets = fetch_gamma_markets()
if markets.empty:
    markets = markets_from_legacy_files()
    LOGS.append({"event": "legacy_market_inputs", "rows": len(markets)})
if markets.empty and SELECTOR_CFG.get("use_mock_if_no_inputs", True):
    markets = mock_relevant_markets()
    LOGS.append({"event": "mock_market_inputs", "rows": len(markets)})

relevant_markets_df = filter_relevant_markets(markets)
if relevant_markets_df.empty and not markets.empty:
    # Existing anomaly files may be geopolitics-like but not keyword-confirmed due abbreviated titles. Keep an auditable fallback.
    relevant_markets_df = markets.head(int(SELECTOR_CFG.get("max_markets", 500))).copy()
    relevant_markets_df["selector_warning"] = "fallback_retained_because_filter_removed_all_rows"
    LOGS.append({"event": "selector_fallback_retained_all", "rows": len(relevant_markets_df)})

csv_path, json_path = write_relevant_markets(relevant_markets_df)
log_path = write_run_log("01_market_selector", LOGS + [{"event": "selector_outputs", "csv_path": str(csv_path), "json_path": str(json_path), "rows": len(relevant_markets_df)}])
print(f"Relevant markets written: {len(relevant_markets_df):,}")
print(csv_path)
print(json_path)
print(log_path)
display(relevant_markets_df.head(20))


Relevant markets written: 21
/Users/R2-D2/Documents/Codex/Falnama/repositories/relevant_markets/relevant_markets_20260615T223937Z.csv
/Users/R2-D2/Documents/Codex/Falnama/repositories/relevant_markets/relevant_markets_20260615T223937Z.json
/Users/R2-D2/Documents/Codex/Falnama/repositories/run_logs/01_market_selector_20260615T223937Z.jsonl


,market_id,market_slug,market_name,event_id,event_slug,condition_id,description,category,tags,volume,liquidity,start_date,close_time,market_url,source
0,923041,will-people-s-party-pple-win-the-most-seats-in-the-2026-thai-legislative-election,Will People’s Party (PPLE) win the most seats in the 2026 Thai legislative election?,103730,None,0x6a79204ace08d896a72cac89c3c2b4c64b90218838c213e9377860596dc77ac5,None,None,None,6.981656e+06,NaN,None,None,https://polymarket.com/event/thai-legislative-election-winner/will-peoples-party-pple-win-the-most-seats-in-the-2026-thai-legislative-election,legacy_ranked_anomalies
1,923042,will-bhumjaithai-party-bjt-win-the-most-seats-in-the-2026-thai-legislative-election,Will Bhumjaithai Party (BJT) win the most seats in the 2026 Thai legislative election?,103730,None,0x91fd3b7cf10e925f169ff49dd657baf9794025b241965fecd0c1b9cbd9c1e115,None,None,None,3.057801e+06,NaN,None,None,https://polymarket.com/event/thai-legislative-election-winner/will-bhumjaithai-party-bjt-win-the-most-seats-in-the-2026-thai-legislative-election,legacy_ranked_anomalies
2,1271753,will-bhumjaithai-party-bjt-finish-in-second-place-by-number-of-seats-in-the-2026-thai-legislative-el,Will Bhumjaithai Party (BJT) finish in second place by number of seats in the 2026 Thai legislative election?,189569,None,0x1a19d02565c09088eb2b5fddb5e014cad5b755d812f36a5b35588158a84f618b,None,None,None,6.372777e+04,NaN,None,None,https://polymarket.com/event/thailand-legislative-election-2nd-place/will-bhumjaithai-party-bjt-finish-in-second-place-by-number-of-seats-in-the-2026-thai-legislative-election,legacy_ranked_anomalies
3,1271752,will-people-s-party-pple-finish-in-second-place-by-number-of-seats-in-the-2026-thai-legislative-elec,Will People’s Party (PPLE) finish in second place by number of seats in the 2026 Thai legislative election?,189569,None,0x1a196362d0cbd39ee1d90cf618bacc2792887f4e36d43d211276929235032ac8,None,None,None,4.235918e+04,NaN,None,None,https://polymarket.com/event/thailand-legislative-election-2nd-place/will-peoples-party-pple-finish-in-second-place-by-number-of-seats-in-the-2026-thai-legislative-election,legacy_ranked_anomalies
4,1272508,will-the-people-s-party-pple-win-between-120-and-134-seats-in-the-2026-thai-legislative-election,Will the People’s Party (PPLE) win between 120 and 134 seats in the 2026 Thai legislative election?,189693,None,0xaed83cf698a5791f232e9e02f91633235fe42c18e7fa38a7c5f01760037b019f,None,None,None,1.437373e+05,NaN,None,None,https://polymarket.com/event/of-seats-won-by-pple-in-2026-thailand-legislative-election/will-the-peoples-party-pple-win-between-120-and-134-seats-in-the-2026-thai-legislative-e...,legacy_ranked_anomalies
5,1272538,will-the-bhumjaithai-party-bjt-win-140-or-more-seats-in-the-2026-thai-legislative-election,Will the Bhumjaithai Party (BJT) win 140 or more seats in the 2026 Thai legislative election?,189699,None,0x9ca4bd854c2d5093d8ad19c99c5eadff83d001ce68c6baa380ead0d62494c004,None,None,None,2.739361e+04,NaN,None,None,https://polymarket.com/event/of-seats-won-by-bjt-in-2026-thailand-legislative-election/will-the-bhumjaithai-party-bjt-win-140-or-more-seats-in-the-2026-thai-legislative-election,legacy_ranked_anomalies
6,1271758,will-chart-thai-pattana-party-ctpp-finish-in-second-place-by-number-of-seats-in-the-2026-thai-legisl,Will Chart Thai Pattana Party (CTPP) finish in second place by number of seats in the 2026 Thai legislative election?,189569,None,0xc52cdfdf348a4ac084c8b08f19f8d69fbe8cfd61cdeca9950b0c3954ba634510,None,None,None,5.350896e+03,0.0,None,None,https://polymarket.com/event/thailand-legislative-election-2nd-place/will-chart-thai-pattana-party-ctpp-finish-in-second-place-by-number-of-seats-in-the-2026-thai-legislative-e...,legacy_ranked_anomalies
7,1271755,will-palang-pracharath-party-pprp-finish-in-second-place-by-number-of-seats-in-the-2026-thai-legisla,Will Palang Pracharath Party (PPRP) finish in second place by number of seats in the 2026 Thai legislative election?,189